# Agent-Based Truck Productivity Simulation

This notebook demonstrates the basic usage of the simulation framework.

In [ ]:
# Import required modules
import sys
sys.path.insert(0, '..')

from src.simulation import Simulation, SimulationConfig
from src.environment import RoadNetwork, NetworkType
from src.utils import Visualizer
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Create Road Network

In [ ]:
# Create a 10x10 grid network
network = RoadNetwork(
    network_type=NetworkType.GRID,
    config={
        "network": {
            "dimensions": [10, 10],
            "edge_length": 1.0
        }
    }
)
network._create_grid_network({"dimensions": [10, 10], "edge_length": 1.0})
network.depots = {"n_0_0", "n_9_9"}

# Print network statistics
stats = network.get_network_stats()
print("Network Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 2. Configure Simulation

In [ ]:
# Define simulation parameters
config = SimulationConfig(
    n_trucks=50,                    # Number of trucks
    duration=480,                   # Duration in minutes (8 hours)
    time_step=1,                    # Time step in minutes
    routing_policy="congestion_aware",
    demand_rate=15,                 # Tasks per hour
    seed=42                         # For reproducibility
)

print("Simulation Configuration:")
print(f"  Trucks: {config.n_trucks}")
print(f"  Duration: {config.duration} minutes")
print(f"  Routing Policy: {config.routing_policy}")

## 3. Run Simulation

In [ ]:
# Create and run simulation
sim = Simulation(network=network, config=config)
metrics = sim.run()

print("\nSimulation Complete!")

## 4. Analyze Results

In [ ]:
# Display key metrics
print("Simulation Results:")
print("-" * 40)
print(f"  Total Deliveries: {metrics.total_deliveries}")
print(f"  Failed Deliveries: {metrics.failed_deliveries}")
print(f"  Total Distance: {metrics.total_distance:.2f} km")
print(f"  Total Fuel: {metrics.total_fuel:.2f} L")
print(f"  Average Speed: {metrics.avg_speed:.2f} km/h")
print(f"  Average Trip Time: {metrics.avg_trip_time:.2f} min")
print(f"  Fleet Efficiency: {metrics.fleet_efficiency:.4f} deliveries/km")

## 5. Visualize Results

In [ ]:
# Create visualizer
viz = Visualizer(metrics)

# Plot productivity timeline
fig = viz.plot_productivity_timeline(metrics)
plt.show()

In [ ]:
# Plot network with final congestion state
congestion = sim.traffic_model.get_congestion()
fig = viz.plot_network(
    sim.network,
    congestion=congestion,
    title="Final Network Congestion State"
)
plt.show()

## 6. Experiment with Different Policies

In [ ]:
# Compare different routing policies
policies = ['shortest', 'fastest', 'congestion_aware', 'adaptive']
results = {}

for policy in policies:
    config.routing_policy = policy
    sim = Simulation(network=network, config=config)
    metrics = sim.run()
    results[policy] = metrics
    print(f"{policy}: {metrics.total_deliveries} deliveries")

# Plot comparison
fig = plot_simulation_comparison(
    list(results.values()),
    labels=policies,
    metrics=['total_deliveries', 'avg_speed'],
    title="Routing Policy Comparison"
)
plt.show()